In [1]:
import digitalhub as dh

project = dh.get_or_create_project("test-xinet")

In [4]:
func = project.get_function("xinet-pose")
BASE_URL = func.list_runs(kind="openinference+serve:run")[0].status.service['url']
MODEL_NAME = 'XiNet-s-pose-224'

In [5]:
import requests
import urllib.request

img_url = 'https://dn721803.ca.archive.org/0/items/2_20200617_202006/1.JPG'
with urllib.request.urlopen(img_url) as response:
    image_bytes = response.read()

    request = {
        "inputs": [
                {
                    "name": "input",
                    "datatype": "UINT8",
                    "shape": [1, len(image_bytes)],
                    "data": list(image_bytes)
                }
            ]    
    }

In [ ]:
response = requests.post(
        f"http://{BASE_URL}/v2/models/{MODEL_NAME}/infer",
        json=request,
        headers={"Content-Type": "application/json"}
    )

print(f"Status Code: {response.status_code}")
data = response.json()

In [11]:
import time
from io import BytesIO
from PIL import Image

image_bytes = data['outputs'][0]['data']

timestamp = time.strftime("%Y%m%d-%H%M%S", time.gmtime())   
output_filename = f"./output/image_{timestamp}.jpg"

# Convert bytes to PIL Image
image_bytes_clean = bytes(image_bytes)
image = Image.open(BytesIO(image_bytes_clean)).convert('RGB')
image.save(output_filename)
